In [ ]:
display(
    dbutils.fs.ls(
        "abfss://bronze@datalakelipti563252.dfs.core.windows.net/SalesLT/"
    )
)

In [ ]:
df = spark.read.parquet(
    "abfss://bronze@datalakelipti563252.dfs.core.windows.net/SalesLT/Address/"
)

display(df)

In [ ]:
from pyspark.sql.functions import from_utc_timestamp, date_format
from pyspark.sql.types import TimestampType
df = df.withColumn("ModifiedDate", date_format(from_utc_timestamp(df["ModifiedDate"].cast(TimestampType()),"UTC"), "yyyy-MM-dd"))
display(df)

In [ ]:
table_name = []
for i in dbutils.fs.ls(
        "abfss://bronze@datalakelipti563252.dfs.core.windows.net/SalesLT/"
    ):
  print(i.name)
  table_name.append(i.name.split('/')[0])

In [ ]:
from pyspark.sql.functions import from_utc_timestamp, date_format
from pyspark.sql.types import TimestampType

for i in table_name:
  path = 'abfss://bronze@datalakelipti563252.dfs.core.windows.net/SalesLT/' + i + '/' + i + '.parquet'
  df = spark.read.format('parquet').load(path)
  column = df.columns

  for col in column:
    if "Date" in col or "date" in col:
      df = df.withColumn(col, date_format(from_utc_timestamp(df[col].cast(TimestampType()), "UTC"), "yyyy-MM-dd"))

  output_path = 'abfss://silver@datalakelipti563252.dfs.core.windows.net/SalesLT/' +i +'/'
  df.write.format('delta').mode("overwrite").save(output_path)

# COMMAND ----------

display(df)

In [ ]:
table_name = []

for i in dbutils.fs.ls('abfss://silver@datalakelipti563252.dfs.core.windows.net/SalesLT/'):
  print(i.name)
  table_name.append(i.name.split('/')[0])


In [ ]:
table_name
for name in table_name:
  path = 'abfss://silver@datalakelipti563252.dfs.core.windows.net/SalesLT/' + name
  print(path)
  df = spark.read.format('delta').load(path)

  # Get the list of column names
  column_names = df.columns

  for old_col_name in column_names:
      # Convert column name from ColumnName to Column_Name format
      new_col_name = "".join(["_" + char if char.isupper() and not old_col_name[i - 1].isupper() else char for i, char in enumerate(old_col_name)]).lstrip("_")
      
      # Change the column name using withColumnRenamed and regexp_replace
      df = df.withColumnRenamed(old_col_name, new_col_name)

  output_path = 'abfss://gold@datalakelipti563252.dfs.core.windows.net/SalesLT/' +name +'/'
  df.write.format('delta').mode("overwrite").save(output_path)
  display(df)